# Day 3 — Classification Fundamentals

**Agri-AI Bootcamp** · Phase 1 of 5: *ML Fundamentals* · Days 1–4

> Train sklearn LogisticRegression to predict Needs Water; evaluate with confusion matrix and ROC-AUC.

**Runtime:** a free CPU runtime is enough for today.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

Logistic regression turns a weighted sum of features into a probability with the sigmoid, then a threshold (0.5 by default) makes the Yes/No call. Cross-entropy punishes confident wrong answers heavily.

### 🌾 Agriculture example

Soil moisture 18%, 34 °C, 40% humidity → P(needs water) = 0.87 → irrigate. Lowering the threshold catches more dry fields at the cost of extra watering.

> **Watch out:** Accuracy alone misleads; read the confusion matrix and ROC-AUC.

## 2. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)

## 3. Data

In [ ]:
# ---- Irrigation decision data ------------------------------------------------
# Course dataset: Kaggle "Crop Recommendation" (N, P, K, pH, rainfall).
# Simulated here with the fields a soil-moisture sensor would actually report.
rng = np.random.default_rng(7)
n = 800
moisture = rng.uniform(5, 45, n)          # %
temp     = rng.uniform(18, 42, n)         # degC
humidity = rng.uniform(20, 95, n)         # %
# ground truth: dry + hot + dry air -> needs water (with noise)
score = -0.35 * moisture + 0.12 * temp - 0.05 * humidity + 5.0
p_true = 1 / (1 + np.exp(-score))
needs_water = (rng.random(n) < p_true).astype(int)
sens = pd.DataFrame(dict(moisture=moisture, temp=temp, humidity=humidity,
                         needs_water=needs_water))
print(sens.needs_water.value_counts().rename("rows").to_string())
sens.head()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
m=LogisticRegression().fit(Xtr,ytr)
print(confusion_matrix(yte,m.predict(Xte)))
print(roc_auc_score(yte,m.predict_proba(Xte)[:,1]))
```

## 5. Worked solution

In [ ]:
Xs = sens[["moisture", "temp", "humidity"]].to_numpy()
ys = sens.needs_water.to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=.25, random_state=0, stratify=ys)

sc = StandardScaler().fit(Xtr)                 # fit on TRAIN only
m  = LogisticRegression().fit(sc.transform(Xtr), ytr)

prob = m.predict_proba(sc.transform(Xte))[:, 1]
pred = (prob >= 0.5).astype(int)

print("confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(yte, pred))
print("\n", classification_report(yte, pred, target_names=["hold", "irrigate"]))
print(f"ROC-AUC {roc_auc_score(yte, prob):.3f}   (accuracy alone = {(pred==yte).mean():.3f})")

# ---- read one field the way the farmer would ---------------------------------
field = np.array([[18, 34, 40]])               # moisture 18%, 34 C, 40% humidity
p = m.predict_proba(sc.transform(field))[0, 1]
print(f"\nfield (18% / 34C / 40%): P(needs water) = {p:.2f} -> "
      f"{'IRRIGATE' if p >= .5 else 'HOLD'}")
print("weights (scaled):", dict(zip(["moisture","temp","humidity"], m.coef_[0].round(2))))

# ---- threshold is a FARMING decision, not a maths one ------------------------
print("\nthr   recall(dry fields caught)  precision(water not wasted)")
for thr in [.3, .4, .5, .6, .7]:
    pr = (prob >= thr).astype(int)
    tp = ((pr == 1) & (yte == 1)).sum(); fp = ((pr == 1) & (yte == 0)).sum()
    fn = ((pr == 0) & (yte == 1)).sum()
    print(f"{thr:<5} {tp/(tp+fn):>10.3f} {tp/max(tp+fp,1):>24.3f}")

fpr, tpr, _ = roc_curve(yte, prob)
plt.plot(fpr, tpr, lw=2); plt.plot([0,1],[0,1],'--',c='grey')
plt.xlabel("false positive rate"); plt.ylabel("true positive rate")
plt.title(f"ROC  AUC={roc_auc_score(yte, prob):.3f}"); plt.show()

## 6. Your turn

1. Missing a dry field costs more than watering a wet one. Pick the threshold you would ship and justify it in one line.
2. Drop `humidity` and re-fit. How much AUC do you lose?
3. Add a squared moisture term and see whether the boundary improves.

In [ ]:
# TODO 1: the threshold you would ship
MY_THR = 0.40   # <- change and defend it
pr = (prob >= MY_THR).astype(int)
print(confusion_matrix(yte, pr), "\nreason: ...")

# TODO 2: two features only
m2 = LogisticRegression().fit(sc.transform(Xtr)[:, :2], ytr)
p2 = m2.predict_proba(sc.transform(Xte)[:, :2])[:, 1]
print(f"AUC without humidity: {roc_auc_score(yte, p2):.3f}")

# TODO 3: add moisture**2 as a feature and compare AUC

## 7. Dataset and references

**Dataset:** [Kaggle: Crop Recommendation (N, P, K, pH, rainfall)](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)

**References**

- [ISLR book](https://www.statlearning.com)
- [StatQuest](https://statquest.org)

**Course material:** ISLR Ch.4; StatQuest Logistic Regression

---
**Next:** Day 4 — Model Evaluation & Regularization